In [2]:
import zipfile
import os

def unzip_file(zip_path, extract_to=None):
    # extract_to가 지정되지 않았으면, zip 파일이 있는 폴더에 풀기
    if extract_to is None:
        extract_to = os.path.splitext(zip_path)[0]  # zip파일과 같은 이름의 폴더 생성

    # 폴더가 없으면 생성
    os.makedirs(extract_to, exist_ok=True)

    # 압축 해제
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
        print(f"압축이 해제되었습니다: {extract_to}")

# 사용 예시
unzip_file("1024_resized_case_study_images.zip")


압축이 해제되었습니다: 1024_resized_case_study_images


In [ ]:
import os

folder_path = "1024_resized_case_study_images/1024_resized_case_study_images"

for filename in os.listdir(folder_path):
    if filename.endswith('00.jpg'):
        file_path = os.path.join(folder_path, filename)
        os.remove(file_path)
        print(f"{filename} 삭제 완료")


In [1]:
import base64
import os
from openai import OpenAI
from PIL import Image
from io import BytesIO
from tqdm import tqdm  # 1. tqdm 임포트
from dotenv import load_dotenv

load_dotenv()

# OpenAI API 키 불러오기
client = OpenAI(api_key=os.getenv("GOODFELLOW_OPENAI_API_KEY_"))

# 입력 및 출력 폴더 경로
input_folder = "1024_resized_case_study_images/1024_resized_case_study_images"
output_folder = "1024_bf_case_image"

# 출력 폴더가 없다면 생성
os.makedirs(output_folder, exist_ok=True)

# 2. 먼저 처리할 .jpg 파일 목록을 만듭니다.
try:
    all_files = os.listdir(input_folder)
    jpg_files = [f for f in all_files if f.lower().endswith(".jpg")]
    
    if not jpg_files:
        print(f"'{input_folder}'에서 처리할 .jpg 파일을 찾을 수 없습니다.")
        exit()
        
    print(f"총 {len(jpg_files)}개의 .jpg 파일 인페인팅을 시작합니다...")

except FileNotFoundError:
    print(f"오류: 입력 폴더 '{input_folder}'를 찾을 수 없습니다.")
    exit()


# 3. tqdm으로 파일 목록을 감싸서 진행 상황을 표시합니다.
for filename in tqdm(jpg_files, desc="🎨 이미지 인페인팅 중"):
    input_path = os.path.join(input_folder, filename)
    output_filename = f"bf_{filename}"
    output_path = os.path.join(output_folder, output_filename)

    # 6. [추가] 출력 파일이 이미 존재하는지 확인합니다.
    if os.path.exists(output_path):
        # 7. [추가] 존재한다면, tqdm.write로 메시지를 남기고 이 파일을 건너뜁니다.
        # tqdm.write를 사용하면 진행률 표시줄을 방해하지 않고 로그를 남길 수 있습니다.
        # 이 줄이 꼭 필요하지는 않지만, 어떤 파일을 건너뛰었는지 알 수 있어 유용합니다.
        # tqdm.write(f"⏭️ 이미 존재함: '{output_filename}' (건너뛰기)") 
        continue  # 다음 파일로 넘어감

    # 4. (참고) 루프 내의 print문은 tqdm 진행 표시줄을 방해하므로 제거합니다.
    # print(f"🛠️ {filename} 인페인팅 중...") 

    try:
        # OpenAI API 호출
        response = client.images.edit(
            model="gpt-image-1",
            image=open(input_path, "rb"),
            prompt=(
                "Remove all furniture from the image and fill the background naturally. "
                "Keep walls, floor, and lighting realistic, photorealistic 4k quality."
            ),
            size="1024x1024"
        )

        # 결과 디코딩 및 저장
        image_base64 = response.data[0].b64_json
        image_bytes = base64.b64decode(image_base64)

        with open(output_path, "wb") as f:
            f.write(image_bytes)

        # 4. (참고) 완료 print문도 제거합니다.
        # print(f"✅ 저장 완료: {output_path}")

    except Exception as e:
        # 5. 오류 발생 시, tqdm.write를 사용해 진행률 표시줄을 깨지 않고 오류를 출력합니다.
        tqdm.write(f"❌ '{filename}' 처리 중 오류 발생: {e}")

print(f"✅ 모든 작업 완료! 파일이 '{output_folder}'에 저장되었습니다.")

총 4431개의 .jpg 파일 인페인팅을 시작합니다...


🎨 이미지 인페인팅 중:  23%|██▎       | 1010/4431 [4:08:12<14:00:44, 14.75s/it]


KeyboardInterrupt: 

In [ ]:
import shutil

# 압축할 폴더 경로
folder_to_zip = '1024_bf_case_image'

# 압축파일 이름 (확장자 없이)
zip_filename = 'bf_case_images_backup'

# 압축 실행 (ZIP 형식)
shutil.make_archive(zip_filename, 'zip', folder_to_zip)

print(f"✅ 압축 완료: {zip_filename}.zip")


In [2]:
import os
import shutil

# 폴더 경로
original_folder = "1024_resized_case_study_images/1024_resized_case_study_images"
edited_folder = "1024_bf_case_image"
output_base = "matched_images"

# 출력 베이스 폴더 생성
os.makedirs(output_base, exist_ok=True)

# 편집된 이미지 목록 가져오기
edited_files = [
    f for f in os.listdir(edited_folder) if f.lower().endswith(".jpg")
]

matched_count = 0

for edited_file in edited_files:
    # bf_ 접두사 제거
    original_name = edited_file.replace("bf_", "", 1)

    original_path = os.path.join(original_folder, original_name)
    edited_path = os.path.join(edited_folder, edited_file)

    # 원본 폴더에 같은 이름의 이미지가 있을 경우만 진행
    if os.path.exists(original_path):
        matched_count += 1

        # 새 폴더 경로 (파일명 기준 폴더)
        new_folder = os.path.join(output_base, os.path.splitext(original_name)[0])
        os.makedirs(new_folder, exist_ok=True)

        # 새 폴더에 복사
        shutil.copy2(original_path, os.path.join(new_folder, original_name))  # 원본
        shutil.copy2(edited_path, os.path.join(new_folder, edited_file))      # 인페인팅 이미지

print(f"✅ 총 {matched_count}개의 이미지 쌍이 복사되어 '{output_base}' 폴더에 저장되었습니다.")


✅ 총 1010개의 이미지 쌍이 복사되어 'matched_images' 폴더에 저장되었습니다.


In [14]:
import pandas as pd

df = pd.read_csv('built_case.csv')

In [ ]:
import os
import pandas as pd

# CSV 파일 읽기
csv_path = 'built_case.csv'  # 적절한 경로로 수정 가능
df = pd.read_csv(csv_path)

# 기준 폴더 경로
base_dir = 'matched_images'

# 모든 하위 폴더 목록 가져오기
subfolders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]

# 각 행에 대해 처리
for _, row in df.iterrows():
    seq = str(row['Seq'])
    text = str(row['output_text'])

    # 일치하는 폴더 찾기 (seq로 시작하는 폴더)
    matched = [folder for folder in subfolders if folder.startswith(seq + "_")]

    for folder_name in matched:
        folder_path = os.path.join(base_dir, folder_name)
        txt_path = os.path.join(folder_path, f"{seq}.txt")  # 파일명은 원하는대로 조정 가능

        # 텍스트 파일 저장
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(text)

        print(f"Saved to: {txt_path}")
